# 06 · Routing — sending a query to the right source
Real assistants have more than one knowledge source (e.g. an FAQ database and a Product database). Routing decides which to search — the pattern from your course's Module 4.

In [ ]:
# A REAL but simple vectorizer: bag-of-words (term frequency). Offline, deterministic,
# and good enough that similar documents (sharing words) get similar vectors — so
# cosine similarity and retrieval metrics are genuinely meaningful and hand-checkable.
#
# NOTE: production RAG uses NEURAL embeddings (e.g. Jina, OpenAI, sentence-transformers)
# that also capture SYNONYMS ("car" ~ "automobile") with no shared words. The MATH below
# (cosine, retrieval, metrics) is identical; only the vectors get smarter. Where a cell
# says "swap in a real embedder", that's the one line that changes.
import numpy as np, re
def tokenize(text): return re.findall(r"[a-z0-9]+", text.lower())
def build_vocab(texts):
    vocab={}
    for t in texts:
        for w in tokenize(t):
            if w not in vocab: vocab[w]=len(vocab)
    return vocab
def vectorize(text, vocab):
    v=np.zeros(len(vocab))
    for w in tokenize(text):
        if w in vocab: v[vocab[w]]+=1.0
    return v
def cosine(a,b):
    na,nb=np.linalg.norm(a),np.linalg.norm(b)
    return float(a@b/(na*nb)) if na and nb else 0.0

## 1. Two sources

In [ ]:
faq = ["how do I get a refund","how long does shipping take","how do I reset my password",
       "how can I contact support"]
products = ["waterproof winter hiking jacket","lightweight breathable running shoes",
            "organic cotton t-shirt five colors","UV blocking sunglasses"]
faq_vocab = build_vocab(faq+products)  # shared vocab for fair comparison
faq_vecs = [vectorize(t, faq_vocab) for t in faq]
prod_vecs= [vectorize(t, faq_vocab) for t in products]

## 2. A simple semantic router
Compare the query to each source's docs; route to whichever source it's most similar to. (A real system might use an LLM classifier — same idea.)

In [ ]:
def best_sim(qv, vecs): return max(cosine(qv,v) for v in vecs)
def route(query):
    qv=vectorize(query, faq_vocab)
    fs, ps = best_sim(qv,faq_vecs), best_sim(qv,prod_vecs)
    return ("FAQ" if fs>=ps else "PRODUCT", round(fs,3), round(ps,3))

for query in ["I need to reset my password","show me a warm waterproof coat",
              "when will my package arrive","do you have cotton shirts"]:
    dest, fs, ps = route(query)
    print(f"  {query:38s} -> {dest:8s} (faq={fs}, prod={ps})")

**Observe:** password/shipping questions route to FAQ; coat/shirt questions route to PRODUCT. Routing keeps each query searching only the relevant source — less noise, better answers.

```
                 +--> [FAQ retriever]  -----+
  query --> ROUTER                          +--> answer
                 +--> [Product retriever] --+
```
**Your turn:** add a third source (e.g. warranty) and a query that should route to it.